In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/sample_submission.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/item_meta.jsonl
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/test.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train_reviews.jsonl


## 셀 1. 라이브러리와 입력 경로 설정

In [2]:
import hashlib
import json
from pathlib import Path
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from threadpoolctl import threadpool_limits

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

competition_dirs = sorted({
    path.parent for path in INPUT.rglob("train.csv")
    if (path.parent / "test.csv").is_file()
})

if len(competition_dirs) != 1:
    raise RuntimeError(
        f"train.csv와 test.csv가 함께 있는 대회 입력을 하나 연결하세요: "
        f"{competition_dirs}"
    )

TRAIN = competition_dirs[0] / "train.csv"
TEST = competition_dirs[0] / "test.csv"

kit_scripts = sorted(INPUT.rglob("teamkit.py"))

if not kit_scripts:
    kit_zips = sorted(INPUT.rglob("amazon_topk_teamkit_20261110.zip"))
    if len(kit_zips) != 1:
        raise RuntimeError("팀 키트 ZIP을 Private Dataset으로 연결하세요.")

    kit_dir = WORK / "teamkit_input"
    with zipfile.ZipFile(kit_zips[0]) as archive:
        archive.extractall(kit_dir)
    kit_scripts = sorted(kit_dir.rglob("teamkit.py"))

if len(kit_scripts) != 1:
    raise RuntimeError(f"팀 키트 입력을 하나 연결하세요: {kit_scripts}")

TEAMKIT = kit_scripts[0]
VAL = TEAMKIT.parent / "validation"

print("대회 데이터:", TRAIN.parent)
print("팀 키트:", TEAMKIT.parent)

대회 데이터: /kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation
팀 키트: /kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit


## 2. 팀 공통 검증 데이터 확인

In [3]:
def run_teamkit(command, **paths):
    args = [sys.executable, str(TEAMKIT), command]
    for name, value in paths.items():
        args.extend([f"--{name.replace('_', '-')}", str(value)])

    result = subprocess.run(args, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())

    return json.loads(result.stdout)

split = run_teamkit("verify-split", dir=VAL)
manifest = json.loads((VAL / "manifest.json").read_text(encoding="utf-8"))

if split["split_id"] != "f0dfad2f4431a876":
    raise ValueError(f"팀 공통 분할 ID와 다릅니다: {split['split_id']}")

train_hash = hashlib.sha256(TRAIN.read_bytes()).hexdigest()
if train_hash != manifest["source_train_sha256"]:
    raise ValueError("대회 입력의 train.csv가 팀 공통 분할의 원본과 다릅니다.")

sys.path.insert(0, str(TEAMKIT.parent))
from teamkit import read_interactions, read_users, top20_unseen, write_predictions

print(split)

{'split_id': 'f0dfad2f4431a876', 'verified': True}


### 3. Truncated SVD 학습·추천 함수 작성
사용자가 상품과 상호작용한 기록이 있으면 행렬에 1을 넣는다. SVD로 압축한 사용자 벡터와 상품 벡터를 곱해 추천 점수를 계산하고, 이미 본 상품을 제외한 뒤 상위 20개를 고른다. 128명씩 나눠 점수를 계산하므로 전체 사용자×상품 점수 행렬을 메모리에 한꺼번에 올리지 않아요.

In [10]:
N_COMPONENTS = 64
SEED = 42
N_ITER = 7
BATCH_SIZE = 128
TOP_K = 20


def predict_svd(
    fit_path,
    users_path,
    output_path,
    *,
    n_components=N_COMPONENTS,
    weighted=False,
):
    fit = pd.read_csv(fit_path, usecols=["user_id", "item_id"])
    user_rows, user_ids = pd.factorize(fit["user_id"], sort=True)
    item_cols, item_ids = pd.factorize(fit["item_id"], sort=True)

    # 사용자–상품 상호작용을 희소 행렬로 구성
    matrix = csr_matrix(
        (np.ones(len(fit), dtype=np.float32), (user_rows, item_cols)),
        shape=(len(user_ids), len(item_ids)),
    )
    matrix.data.fill(1.0)  # 같은 사용자–상품 쌍이 반복돼도 1로 처리
    matrix.sort_indices()

    if weighted:
        # 상품별로 상호작용한 고유 사용자 수를 계산
        item_user_counts = np.bincount(
            matrix.indices,
            minlength=matrix.shape[1],
        )
        item_weights = (1.0 / np.sqrt(item_user_counts)).astype(np.float32)

        # 상호작용이 있는 칸에 1 / sqrt(상품별 사용자 수)를 적용
        matrix.data *= item_weights[matrix.indices]

    model = TruncatedSVD(
        n_components=n_components,
        n_iter=N_ITER,
        random_state=SEED,
    )

    seen, popularity = read_interactions(fit_path)
    catalog = set(popularity)
    fallback = sorted(catalog, key=lambda item: (-popularity[item], int(item)))
    users = read_users(users_path)

    row_by_user = {str(user): row for row, user in enumerate(user_ids)}
    item_names = np.asarray(item_ids.astype(str))
    predictions = {}

    with threadpool_limits(limits=2, user_api="blas"):
        user_factors = model.fit_transform(matrix)

        for start in range(0, len(users), BATCH_SIZE):
            batch = users[start:start + BATCH_SIZE]
            known = [
                (user, row_by_user[user])
                for user in batch
                if user in row_by_user
            ]

            if known:
                scores = user_factors[[row for _, row in known]] @ model.components_

                for (user, row), score in zip(known, scores):
                    # 학습 데이터에서 이미 본 상품을 추천 후보에서 제외
                    row_start = matrix.indptr[row]
                    row_end = matrix.indptr[row + 1]
                    seen_cols = matrix.indices[row_start:row_end]
                    score[seen_cols] = -np.inf

                    top = np.argpartition(score, -TOP_K)[-TOP_K:]
                    top = top[np.lexsort((top, -score[top]))]
                    candidates = item_names[top[np.isfinite(score[top])]]

                    predictions[user] = top20_unseen(
                        user, candidates, seen, fallback, catalog
                    )

            # 학습 기록이 없는 사용자에게는 인기 상품으로 추천
            for user in batch:
                if user not in row_by_user:
                    predictions[user] = top20_unseen(
                        user, (), seen, fallback, catalog
                    )

    write_predictions(output_path, users, predictions)
    print(f"사용자 {len(users):,}명의 추천 저장 → {output_path}")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


### 셀 4. 공통 검증 실행 및 실험 기록

In [9]:
VAL_PRED = WORK / f"svd_k{N_COMPONENTS}_seed{SEED}_val.csv"

predict_svd(
    fit_path=VAL / "val_train.csv",
    users_path=VAL / "val_users.csv",
    output_path=VAL_PRED,
)

result = run_teamkit(
    "score",
    pred=VAL_PRED,
    fit=VAL / "val_train.csv",
    truth=VAL / "val_truth.csv",
)
print(json.dumps(result, indent=2))

record = pd.DataFrame([{
    "split_id": split["split_id"],
    "model": f"TruncatedSVD_k{N_COMPONENTS}",
    "model_seed": SEED,
    "training_interactions": "validation/val_train.csv",
    "extra_inputs": "none",
    "recall_at_20": result["recall_at_20"],
    "hits": result["hits"],
    "denominator": result["denominator"],
    "notes": f"binary interactions; n_iter={N_ITER}",
}])
record.to_csv(WORK / f"svd_k{N_COMPONENTS}_seed{SEED}_experiment.csv", index=False)
display(record)

사용자 36,043명의 추천 저장 → /kaggle/working/svd_k64_seed42_val.csv
{
  "recall_at_20": 0.02266474996533884,
  "hits": 3106,
  "denominator": 137041,
  "users": 36043,
  "valid_submission_format": true
}


,split_id,model,model_seed,training_interactions,extra_inputs,recall_at_20,hits,denominator,notes
0,f0dfad2f4431a876,TruncatedSVD_k64,42,validation/val_train.csv,none,0.022665,3106,137041,binary interactions; n_iter=7


####  Truncated SVD 설정 변경해보기

1.
N_COMPONENTS를 32, 128로 바꿔 기존 64와 비교	
-> 잠재 차원 수에 따라 추천 순위가 개선되는지 확인

2.
가장 점수가 높았던 차원에서 인기 상품의 입력 가중치를 낮춰 한 번 더 실행
(여러 사용자가 본 상품의 영향력을 줄이면 개인별 추천이 개선되는지 확인)

### 추가 셀 A — 가중치 없이 32·64·128차원 비교
기존 64차원 예측 파일이 남아 있으면 그 파일을 다시 채점하고, 없으면 64차원도 다시 실행해요. 세 설정 모두 같은 공통 검증 분할과 시드를 사용해요.

In [11]:
def evaluate_svd(n_components, *, weighted=False, existing_prediction=None):
    variant = "inv_sqrt_pop" if weighted else "binary"
    model_name = f"TruncatedSVD_k{n_components}_{variant}"
    pred_path = WORK / f"{model_name}_seed{SEED}_val.csv"

    if existing_prediction is not None and existing_prediction.is_file():
        pred_path = existing_prediction
        print(f"기존 예측 파일 사용 → {pred_path.name}")
    else:
        print(f"검증 실행 → {model_name}")
        predict_svd(
            fit_path=VAL / "val_train.csv",
            users_path=VAL / "val_users.csv",
            output_path=pred_path,
            n_components=n_components,
            weighted=weighted,
        )

    result = run_teamkit(
        "score",
        pred=pred_path,
        fit=VAL / "val_train.csv",
        truth=VAL / "val_truth.csv",
    )

    return {
        "split_id": split["split_id"],
        "model": model_name,
        "model_seed": SEED,
        "training_interactions": "validation/val_train.csv",
        "extra_inputs": "none",
        "recall_at_20": result["recall_at_20"],
        "hits": result["hits"],
        "denominator": result["denominator"],
        "notes": f"weighting={variant}; n_iter={N_ITER}",
        "n_components": n_components,
    }


baseline_64 = WORK / f"svd_k64_seed{SEED}_val.csv"

rank_results = pd.DataFrame([
    evaluate_svd(32),
    evaluate_svd(64, existing_prediction=baseline_64),
    evaluate_svd(128),
]).sort_values(["recall_at_20", "n_components"], ascending=[False, True])

best_rank = int(rank_results.iloc[0]["n_components"])

display(rank_results[["n_components", "recall_at_20", "hits"]])
print(f"가중치를 적용할 차원 → {best_rank}")

검증 실행 → TruncatedSVD_k32_binary
사용자 36,043명의 추천 저장 → /kaggle/working/TruncatedSVD_k32_binary_seed42_val.csv
기존 예측 파일 사용 → svd_k64_seed42_val.csv
검증 실행 → TruncatedSVD_k128_binary
사용자 36,043명의 추천 저장 → /kaggle/working/TruncatedSVD_k128_binary_seed42_val.csv


,n_components,recall_at_20,hits
0,32,0.024584,3369
1,64,0.022665,3106
2,128,0.020585,2821


가중치를 적용할 차원 → 32


### 추가 셀 B — 가장 점수가 높았던 차원에 가중치 적용
선택된 차원에서 가중치만 바꿔 다시 학습해요. 예를 들어 학습 데이터에서 100명이 상호작용한 상품의 가중치는 0.1, 4명이 상호작용한 상품의 가중치는 0.5가 돼요.

In [12]:
weighted_result = evaluate_svd(best_rank, weighted=True)
binary_result = rank_results.iloc[0]

all_results = pd.concat(
    [rank_results, pd.DataFrame([weighted_result])],
    ignore_index=True,
).sort_values("recall_at_20", ascending=False)

display(all_results[["model", "recall_at_20", "hits"]])

delta_recall = weighted_result["recall_at_20"] - binary_result["recall_at_20"]
delta_hits = weighted_result["hits"] - int(binary_result["hits"])

print(f"같은 {best_rank}차원에서 가중치 적용 전후 차이")
print(f"Recall@20 {delta_recall:+.8f} | 적중 {delta_hits:+,}건")
print("인기 상품 기준선 Recall@20 0.02283988 | 적중 3,130건")

experiment_path = WORK / "svd_rank_weighting_experiments.csv"
all_results.drop(columns="n_components").to_csv(experiment_path, index=False)
print(f"팀 공통 양식으로 저장 → {experiment_path}")

검증 실행 → TruncatedSVD_k32_inv_sqrt_pop
사용자 36,043명의 추천 저장 → /kaggle/working/TruncatedSVD_k32_inv_sqrt_pop_seed42_val.csv


,model,recall_at_20,hits
0,TruncatedSVD_k32_binary,0.024584,3369
1,TruncatedSVD_k64_binary,0.022665,3106
2,TruncatedSVD_k128_binary,0.020585,2821
3,TruncatedSVD_k32_inv_sqrt_pop,0.016856,2310


같은 32차원에서 가중치 적용 전후 차이
Recall@20 -0.00772761 | 적중 -1,059건
인기 상품 기준선 Recall@20 0.02283988 | 적중 3,130건
팀 공통 양식으로 저장 → /kaggle/working/svd_rank_weighting_experiments.csv


### 셀 5. 최종 제출 파일 생성 및 검사

In [15]:
SUBMISSION = WORK / "submission.csv"

predict_svd(
    fit_path=TRAIN,
    users_path=TEST,
    output_path=SUBMISSION,
    n_components=32,
    weighted=False,
)

check = run_teamkit("check", pred=SUBMISSION, train=TRAIN, test=TEST)
print(json.dumps(check, indent=2))

사용자 36,043명의 추천 저장 → /kaggle/working/submission.csv
{
  "valid": true,
  "users": 36043,
  "items_per_user": 20
}
